### **Imports**

In [ ]:
import os
import zarr
import numpy as np
# import dask.array as da

# from scipy.ndimage import median_filter
# from skimage.restoration import rolling_ball
# from scipy.ndimage import gaussian_filter
# from skimage.exposure import equalize_adapthist

import cupy as cp
from cucim.skimage.filters import median as gpu_median_filter
from cucim.skimage.exposure import equalize_adapthist as gpu_equalize_adapthist
from cucim.skimage.restoration import rolling_ball as gpu_rolling_ball

TRAIN_DIR = '../train'
TEST_DIR = '../test'
RANGE = 1

videos = [
    # '44b6_0113de3b', 
    # '44b6_0b24845f', 
    # '6bba_05b6850b', 
    # '6bba_05db0fb1'
]

guinea_pig = np.asarray(zarr.open(f'{TEST_DIR}/{videos[0]}.zarr/0'))[:RANGE]
display(guinea_pig.shape)
display(guinea_pig.dtype)

(1, 64, 256, 256)

dtype('<u2')

### **filter testing**

In [47]:
# def do_median_filt(img, size=3, inplace=False):
#     if inplace:
#         for i in range(img.shape[0]):
#             img[i] = median_filter(img[i], size=size)
#         return

#     all_vols = []
#     for i in range(img.shape[0]):
#         vol = img[i]
#         all_vols.append(median_filter(vol, size=size))

#     return np.stack(all_vols)

def do_median_filt_gpu (img, size=3, inplace=False):
    if inplace:
        for i in range(img.shape[0]):
            img[i] = gpu_median_filter(img[i], footprint=size)
        return

    all_vols = []
    for i in range(img.shape[0]):
        vol = img[i]
        all_vols.append(gpu_median_filter(vol, footprint=size))

    return np.stack(all_vols)


In [48]:
# def do_rolling_ball(img, radius=20, inplace=False):
#     if not inplace:
#         corrected = np.empty_like(img, dtype=np.float32)
#     for i in range(img.shape[0]):
#         vol = img[i]
#         background = rolling_ball(vol, radius=radius)
#         filtered = vol - background
#         if inplace:
#             vol[:] = filtered
#         else:
#             corrected[i] = filtered
#     if not inplace:
#         return corrected

def do_rolling_ball_gpu(img, radius=20, inplace=False):
    if not inplace:
        corrected = np.empty_like(img, dtype=np.float32)
    for i in range(img.shape[0]):
        vol = img[i]
        background = gpu_rolling_ball(vol, radius=radius)
        filtered = vol - background
        if inplace:
            vol[:] = filtered
        else:
            corrected[i] = filtered
    if not inplace:
        return corrected


In [49]:
# def do_clahe(img, kernel_size=8, clip_limit=0.01, inplace=False):
#     if not inplace:
#         clahe = np.empty_like(img, dtype=np.float32)
   
#     for i in range(img.shape[0]):
#         vol = img[i]
#         vmin = vol.min()
#         vmax = vol.max()
#         normalized = (vol - vmin) / (vmax - vmin)

#         filtered = equalize_adapthist(
#             normalized,
#             kernel_size=kernel_size,
#             clip_limit=clip_limit,
#         )
#         filtered = filtered * (vmax - vmin) + vmin
       
#         if inplace:
#             vol[:] = filtered
#         else:
#             clahe[i] = filtered
    
#     if not inplace:
#         return clahe

def do_clahe_gpu(img, kernel_size=8, clip_limit=0.01, inplace=False):
    if not inplace:
        clahe = np.empty_like(img, dtype=np.float32)
   
    for i in range(img.shape[0]):
        vol = img[i]
        vmin = vol.min()
        vmax = vol.max()
        normalized = (vol - vmin) / (vmax - vmin)

        filtered = gpu_equalize_adapthist(
            normalized,
            kernel_size=kernel_size,
            clip_limit=clip_limit,
        )
        filtered = filtered * (vmax - vmin) + vmin
       
        if inplace:
            vol[:] = filtered
        else:
            clahe[i] = filtered
    
    if not inplace:
        return clahe

In [50]:
def apply_gamma(img, gamma=2.0, inplace=False):
    vmin = img.min()
    vmax = img.max()

    normalized = (img - vmin) / (vmax - vmin)
    corrected = normalized ** gamma
    result = corrected * (vmax - vmin) + vmin

    if inplace:
        img[...] = result
        return

    return result

In [51]:
# result = guinea_pig.copy()
# do_median_filt(result, size=5, inplace=True) #5s per frame
# do_rolling_ball(result, radius=20, inplace=True) #350s per frame
# do_clahe(result, kernel_size=8, clip_limit=0.01, inplace=True) #1s per frame
# resultx = apply_gamma(result, gamma=3.5)

In [52]:
result2 = cp.asarray(guinea_pig, dtype=cp.float32)

In [53]:
do_median_filt_gpu(result2, size=(6,6,6), inplace=True) #1s per frame

In [54]:
do_rolling_ball_gpu(result2, radius=20, inplace=True) #5s per frame

In [55]:
do_clahe_gpu(result2, kernel_size=8, clip_limit=0.01, inplace=True) #5s per frame

In [ ]:
apply_gamma(result2, gamma=1.5, inplace=True)

### **Full loop**

In [58]:
# videos = [
#     '44b6_33b596bf', '44b6_40c45f5a', '44b6_551a5dba',
#     '44b6_87bba6c4', '44b6_abf82518', '44b6_c96cfa10',
#     '44b6_eb2880fc', '6bba_1f58c2f6', '6bba_55c70843',
#     '6bba_61dd1e0d', '6bba_6ca87370', '6bba_6feeb0b1',
#     '6bba_80d12824', '6bba_9e23430b', '6bba_debd7bfa'
# ]


In [ ]:
# import os
# import zarr
# from datetime import datetime

# os.makedirs("../train_processed", exist_ok=True)

# start_time = datetime.now()
# print(f"[{start_time:%H:%M:%S}] Started preprocessing")

# for vid in videos:
#     print(f"[{datetime.now():%H:%M:%S}] Processing {vid}")
#     src = zarr.open(f"../train/{vid}.zarr/0", mode="r")
#     result = src[:].copy()

#     print(f"[{datetime.now():%H:%M:%S}]   Median filter")
#     do_median_filt(result, size=5, inplace=True)
#     print(f"[{datetime.now():%H:%M:%S}]   Rolling ball")
#     do_rolling_ball(result, radius=20, inplace=True)
#     print(f"[{datetime.now():%H:%M:%S}]   CLAHE")
#     do_clahe(result, kernel_size=8, clip_limit=0.01, inplace=True)

#     print(f"[{datetime.now():%H:%M:%S}]   Saving")
#     dst = zarr.open(
#         f"../train_processed/{vid}.zarr/0",
#         mode="w",
#         shape=src.shape,
#         chunks=src.chunks,
#         dtype=result.dtype,
#     )
#     dst[:] = result

#     print(f"[{datetime.now():%H:%M:%S}] Saved: ../train_processed/{vid}.zarr/0")

# end_time = datetime.now()
# print("="*60)
# print(f"[{end_time:%H:%M:%S}] Finished preprocessing")
# print(f"Total time: {end_time - start_time}")

### **Napari**

In [ ]:
import napari
import zarr
import numpy as np

VIDEOS = [
    "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba",
    # "44b6_87bba6c4", "44b6_abf82518", "44b6_c96cfa10",
    # "44b6_eb2880fc", "6bba_1f58c2f6", "6bba_55c70843",
    # "6bba_61dd1e0d", "6bba_6ca87370", "6bba_6feeb0b1",
    # "6bba_80d12824", "6bba_9e23430b", "6bba_debd7bfa"
]

for vid in VIDEOS:
    orig = np.asarray(zarr.open(f'../train/{vid}.zarr/0'))
    proc = np.asarray(zarr.open(f'../train_processed/{vid}.zarr/0'))
        
    viewer = napari.Viewer(ndisplay=3)

    viewer.add_image(
        orig,
        name='original',
        scale=(1.0, 4.0, 1.0, 1.0),
        colormap='magma',
        gamma=1.0,
        blending='additive'
    )
    viewer.add_image(
        proc,
        name='processed',
        scale=(1.0, 4.0, 1.0, 1.0),
        colormap='magma',
        gamma=1.0,
        blending='additive'
    )

    viewer.axes.visible = True
    viewer.axes.colored = True
    viewer.axes.labels = True
    viewer.axes.dashed = False
    viewer.grid.enabled = True
    # viewer.grid.shape = (1, 3)

    napari.run()

could not determine DPI


KeyboardInterrupt: 

: 